# 01 — Explore Santosh (seed 42)

**Exploration only.** Import the installed package; do **not** put production train/serve logic here.

Promoted code lives under `src/retention_radar/`. Published charts: `results/plots/`.
Published narrative: `results/SANTOSH_ANALYSIS.md` · `results/BENCHMARKS.md`.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from retention_radar import config
from retention_radar.data.generate import santosh_profile
from retention_radar.serving.infer import load_payload, predict_user
from retention_radar.training.calibrate import load_calibrator

print("PROJECT_ROOT:", config.PROJECT_ROOT)
print("SEED models:", config.SEED_MODELS_DIR)
print("MODEL_PATH:", config.MODEL_PATH)

In [ ]:
profile = {k: v for k, v in santosh_profile().items() if k != "churned"}
print(json.dumps(profile, indent=2)[:500], "...")

santosh_path = config.SANTOSH_JSON
print("Committed JSON:", santosh_path, "exists=", santosh_path.exists())

In [ ]:
bundle = load_payload(config.SEED_MODELS_DIR / "churn_xgb.joblib")
cal = load_calibrator(config.SEED_MODELS_DIR / "calibrator.joblib")
result = predict_user(profile, bundle, calibrator=cal)
print(
    f"raw={result['churn_probability_raw']:.4f}  "
    f"cal={result['churn_probability_calibrated']:.4f}  "
    f"band={result['risk_band']}"
)
print("Expected (published): raw≈0.043  cal≈0.016  band=low")

## Results / plots (read-only pointers)

- `results/SANTOSH_ANALYSIS.md` — dual-world Santosh write-up
- `results/BENCHMARKS.md` — honest ladder
- `results/plots/` — ROC / PR / calibration charts for the narrative
- `results/santosh_decision_packet.sample.json` — sample HITL packet

Streamlit UI (`make ui`) loads the **same** committed models — no fit on page load.